# Descarte Fácil — previsão de demanda de remoção de grandes objetos

**Pergunta:** é possível estimar a quantidade mensal de solicitações de remoção de grandes objetos por distrito, usando os registros públicos do SP156?

**Fonte:** Portal de Dados Abertos da Prefeitura de São Paulo — [Dados do SP156](https://dados.prefeitura.sp.gov.br/dataset/dados-do-sp156).

> O SP156 registra solicitações feitas pelos cidadãos. O modelo estima demanda registrada, não todos os descartes irregulares existentes na cidade e nem a quantidade real de resíduos.

In [ ]:
!pip -q install pandas numpy matplotlib seaborn scikit-learn xgboost requests

In [ ]:
import io, requests, pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import ParameterGrid
from xgboost import XGBRegressor

# CSV oficial mais recente identificado no portal (3º trimestre de 2026).
# Se a Prefeitura alterar o endereço, obtenha o novo link na página do conjunto.
DATA_URL = "https://dados.prefeitura.sp.gov.br/dataset/0aecfa2b-aa3a-40d4-8183-0d4351b7fd0a/resource/f67fcc12-8da4-485f-92a3-d2af40cc771e/download/arquivo-final-3-trim-26.csv"
r = requests.get(DATA_URL, timeout=180)
r.raise_for_status()
raw = pd.read_csv(io.BytesIO(r.content), sep=None, engine="python", encoding_errors="replace")
print(raw.shape)
display(raw.head())
print(raw.columns.tolist())

### Decisões metodológicas

A coluna `Data de Abertura` segue o padrão **MM/DD/YYYY**, confirmado por valores como `07/25/2026`. A unidade geográfica disponível é **Distrito** (não Subprefeitura). Como o arquivo atual contém somente três meses, o experimento usa uma divisão cronológica simples: primeiro mês para treino, segundo para validação e terceiro para teste. Não usamos defasagens como atributos, pois isso eliminaria o primeiro mês e deixaria apenas dois meses para treino/avaliação. O teste é reservado até a avaliação final.

## 1. Preparação e seleção do serviço

As colunas utilizadas são Data de Abertura, Serviço e Distrito. O nome das colunas é conferido no próprio arquivo antes do processamento.

In [ ]:
# As colunas reais do CSV foram conferidas: "Data de Abertura", "Serviço" e "Distrito".
# O arquivo não possui uma coluna de Subprefeitura; a unidade territorial será Distrito.
date_col = "Data de Abertura"
service_col = "Serviço"
district_col = "Distrito"

missing = [c for c in [date_col, service_col, district_col] if c not in raw.columns]
if missing:
    raise ValueError(f"Colunas esperadas não encontradas: {missing}. Confira raw.columns.")

print("Data:", date_col, "| Serviço:", service_col, "| Unidade territorial:", district_col)


In [ ]:
df = raw.copy()

# O formato foi confirmado pelos próprios valores do CSV: por exemplo, 07/25/2026.
# Como 25 não pode ser mês, o padrão é mês/dia/ano (MM/DD/YYYY), não dia/mês/ano.
df[date_col] = pd.to_datetime(df[date_col].astype(str).str.strip(),
                              format="%m/%d/%Y", errors="coerce")
valid_dates = df[date_col].notna().mean()
print(f"Proporção de datas interpretadas: {valid_dates:.1%}")
if valid_dates < 0.95:
    raise ValueError("Menos de 95% das datas foram interpretadas. Verifique se o formato do CSV mudou.")

df = df.dropna(subset=[date_col, district_col, service_col]).copy()

# Seleção inicial por termos relacionados a remoção de grandes objetos.
print(df[service_col].astype(str).value_counts().head(40).to_string())


## 2. Agregação temporal e variável-alvo

A unidade de análise é **distrito-mês**. O alvo é o número de solicitações relacionadas à remoção de grandes objetos em cada distrito no mês. Meses sem solicitações registradas são preenchidos com zero.

In [ ]:
TERMS = ["entulho e grandes objetos", "remoção de grandes objetos", "remocao de grandes objetos"]
service_text = df[service_col].astype(str).str.lower()
selected = df[service_text.apply(lambda x: any(t in x for t in TERMS))].copy()
if selected.empty:
    raise ValueError("Nenhum registro encontrado. Ajuste TERMS com o nome exato exibido em value_counts().")

selected["mes"] = selected[date_col].dt.to_period("M").dt.to_timestamp()
agg = (selected.groupby([district_col, "mes"]).size()
       .rename("solicitacoes").reset_index()
       .rename(columns={district_col:"regiao"}))
# Painel balanceado: meses sem solicitação observada recebem zero.
regions = agg["regiao"].dropna().unique()
months = pd.date_range(agg["mes"].min(), agg["mes"].max(), freq="MS")
panel = pd.MultiIndex.from_product([regions, months], names=["regiao","mes"]).to_frame(index=False)
panel = panel.merge(agg, on=["regiao","mes"], how="left")
panel["solicitacoes"] = panel["solicitacoes"].fillna(0).astype(int)
display(panel.head())
print("Período:", panel.mes.min(), "a", panel.mes.max(), "| linhas:", len(panel))

## 3. Atributos e divisão temporal

Usamos atributos de calendário e identificação do distrito. O histórico disponível tem apenas três meses; por isso, não incluímos lags: com lag de um mês, sobrariam somente dois meses e não seria possível manter treino, validação e teste independentes. A divisão cronológica usa o primeiro mês para treino, o segundo para validação e o terceiro para teste.

In [ ]:
panel = panel.sort_values(["regiao","mes"]).copy()
panel["ano"] = panel["mes"].dt.year
panel["mes_num"] = panel["mes"].dt.month
panel["trimestre"] = panel["mes"].dt.quarter

# Não usar lags neste recorte de apenas 3 meses: isso reduziria o painel
# a 2 meses e impediria separar treino, validação e teste sem sobreposição.
model_df = panel[["regiao", "mes", "solicitacoes", "ano", "mes_num", "trimestre"]].copy()

X = pd.get_dummies(
    model_df[["regiao","ano","mes_num","trimestre"]],
    columns=["regiao"], drop_first=False, dtype=int
)
y = model_df["solicitacoes"]

months = np.array(sorted(model_df["mes"].unique()))
if len(months) < 3:
    raise ValueError(
        f"São necessários pelo menos 3 meses completos para treino, validação e teste independentes. "
        f"Foram encontrados {len(months)}: {months}."
    )

# Três blocos temporais distintos: primeiro mês, segundo mês e terceiro mês.
train_months = months[:1]
val_months = months[1:2]
test_months = months[2:]

train_mask = model_df["mes"].isin(train_months)
val_mask = model_df["mes"].isin(val_months)
test_mask = model_df["mes"].isin(test_months)

X_train, y_train = X.loc[train_mask], y.loc[train_mask]
X_val, y_val = X.loc[val_mask], y.loc[val_mask]
X_trainval, y_trainval = X.loc[train_mask | val_mask], y.loc[train_mask | val_mask]
X_test, y_test = X.loc[test_mask], y.loc[test_mask]

print("Treino:", train_months[0], "| linhas:", len(X_train))
print("Validação:", val_months[0], "| linhas:", len(X_val))
print("Teste:", test_months[0], "| linhas:", len(X_test))
if min(len(X_train), len(X_val), len(X_test)) == 0:
    raise ValueError("Um dos conjuntos ficou vazio. Confira a cobertura mensal do arquivo.")

## 4. Treinamento e comparação

Comparamos uma linha de base (repetir no mês de teste o valor observado no mês anterior) com Random Forest e XGBoost. Os hiperparâmetros dos modelos são selecionados somente na validação. Depois, os modelos são reajustados com treino + validação e avaliados uma única vez no teste.

Métricas: MAE, RMSE e R². O MAE representa o erro absoluto médio em número de solicitações por distrito.

In [ ]:
# Baseline: repetir para cada distrito o valor do mês imediatamente anterior ao teste.
previous_month = test_months[0] - pd.offsets.MonthBegin(1)
baseline_frame = model_df.loc[model_df["mes"].eq(previous_month), ["regiao", "solicitacoes"]]
test_frame = model_df.loc[test_mask, ["regiao", "solicitacoes"]].merge(
    baseline_frame.rename(columns={"solicitacoes":"previsao_baseline"}),
    on="regiao", how="left"
)
if test_frame["previsao_baseline"].isna().any():
    raise ValueError("Não foi possível construir o baseline para todos os distritos do teste.")
baseline = test_frame["previsao_baseline"].to_numpy()
results, fitted = [], {}
results.append({
    "modelo": "Baseline (mês anterior)",
    "MAE": mean_absolute_error(y_test, baseline),
    "RMSE": mean_squared_error(y_test, baseline)**0.5,
    "R2": r2_score(y_test, baseline)
})

rf_grid = list(ParameterGrid({
    "n_estimators": [100, 200],
    "max_depth": [3, 6, None],
    "min_samples_leaf": [1, 3]
}))
best_rf_params, best_rf_mae = None, float("inf")
for params in rf_grid:
    candidate = RandomForestRegressor(random_state=42, n_jobs=-1, **params)
    candidate.fit(X_train, y_train)
    val_pred = candidate.predict(X_val)
    val_mae = mean_absolute_error(y_val, val_pred)
    if val_mae < best_rf_mae:
        best_rf_params, best_rf_mae = params, val_mae

rf = RandomForestRegressor(random_state=42, n_jobs=-1, **best_rf_params)
rf.fit(X_trainval, y_trainval)
fitted["Random Forest"] = rf
rf_pred = rf.predict(X_test)
results.append({
    "modelo": f"Random Forest {best_rf_params}",
    "MAE": mean_absolute_error(y_test, rf_pred),
    "RMSE": mean_squared_error(y_test, rf_pred)**0.5,
    "R2": r2_score(y_test, rf_pred)
})

xgb_grid = list(ParameterGrid({
    "n_estimators": [100, 200],
    "max_depth": [2, 3],
    "learning_rate": [0.03, 0.08]
}))
best_xgb_params, best_xgb_mae = None, float("inf")
for params in xgb_grid:
    candidate = XGBRegressor(
        random_state=42, objective="reg:squarederror", n_jobs=-1, **params
    )
    candidate.fit(X_train, y_train)
    val_pred = candidate.predict(X_val)
    val_mae = mean_absolute_error(y_val, val_pred)
    if val_mae < best_xgb_mae:
        best_xgb_params, best_xgb_mae = params, val_mae

xgb = XGBRegressor(
    random_state=42, objective="reg:squarederror", n_jobs=-1, **best_xgb_params
)
xgb.fit(X_trainval, y_trainval)
fitted["XGBoost"] = xgb
xgb_pred = xgb.predict(X_test)
results.append({
    "modelo": f"XGBoost {best_xgb_params}",
    "MAE": mean_absolute_error(y_test, xgb_pred),
    "RMSE": mean_squared_error(y_test, xgb_pred)**0.5,
    "R2": r2_score(y_test, xgb_pred)
})

metrics = pd.DataFrame(results).sort_values("MAE")
display(metrics)

best_name = metrics.loc[metrics["modelo"].str.startswith(("Random Forest", "XGBoost")), "modelo"].iloc[0]
best_model = fitted["Random Forest"] if best_name.startswith("Random Forest") else fitted["XGBoost"]
best_pred = best_model.predict(X_test)

In [ ]:
plt.figure(figsize=(7,5))
plt.scatter(y_test, best_pred, alpha=.35)
lim = max(float(y_test.max()), float(np.max(best_pred)))
plt.plot([0,lim],[0,lim],"--")
plt.xlabel("Solicitações reais")
plt.ylabel("Solicitações previstas")
plt.title("Valores reais x previstos — modelo selecionado")
plt.tight_layout()
plt.show()

if hasattr(best_model, "feature_importances_"):
    imp = pd.Series(best_model.feature_importances_, index=X.columns).sort_values().tail(12)
    imp.plot(kind="barh", figsize=(8,5), title="Atributos mais importantes")
    plt.tight_layout()
    plt.show()

## 5. Interpretação para o cidadão

Após executar o notebook, registre nesta seção:
- Melhor modelo e hiperparâmetros: **[preencher com a tabela]**.
- MAE no teste: **[preencher]** solicitações por distrito.
- RMSE e R²: **[preencher]**.
- Comparação com o baseline: **[informar se o modelo superou a previsão do mês anterior]**.
- Atributos mais relevantes: **[preencher com o gráfico, se aplicável]**.

**Aplicação no Descarte Fácil:** se o modelo superar o baseline, sua estimativa poderá servir como indicador exploratório para orientar comunicação e planejamento em regiões com maior demanda registrada. Não deve ser apresentada como previsão certa nem substituir o calendário oficial do Cata-Bagulho.

**Limitações:** o arquivo analisado cobre somente três meses (julho a setembro de 2026). Assim, há apenas um mês para treino, um para validação e um para teste, o que torna a avaliação muito frágil e impede conclusões sobre desempenho futuro. O SP156 registra solicitações, não todos os descartes; a classificação dos serviços pode mudar; e a granularidade por distrito não permite inferências por CEP ou endereço. Para um modelo operacional, será necessário obter uma série histórica mais longa e repetir a avaliação em vários períodos temporais.

**Nota:** não interprete R² isoladamente. Compare MAE e RMSE com o baseline e considere o tamanho reduzido do teste (um único mês).